# Lesson 01 - AIエージェント入門

<strong>初心者向けAIエージェント</strong>コースの最初のレッスンへようこそ！

<strong>AIエージェント</strong>とは、大規模言語モデル（LLM）を推論エンジンとして使用し、ユーザーに代わって目的を達成するために実世界で<em>行動</em>を取るプログラムのことです。例えば、APIを呼び出したり、データベースに問い合わせたり、コードを実行したりします。

このノートブックでは、最初のエージェントである <strong>旅行代理エージェント</strong> を構築します。休暇先を推薦するエージェントです。進める中で、以下を学びます：

1. <strong>Microsoft Agent Framework</strong>を使ってMicrosoft Foundry Agent Serviceに接続する方法。
2. エージェントに<strong>ツール</strong>（呼び出せる純粋なPython関数）を与える方法。
3. エージェントを実行し、その応答を確認する方法。
4. エージェントの応答をトークン単位でストリームする方法。


## Setup

Before running this notebook, make sure you have:

1. **A Microsoft Foundry project** with a deployed chat model (e.g. `gpt-4.1-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 最初のエージェントの作成

エージェントには2つのものが必要です：

- <strong>指示</strong>：それが<em>誰であるか</em>、<em>どのように振る舞うか</em>を伝えるもの（システムプロンプト）。
- <strong>ツール</strong> — エージェントが情報を取得したり操作を実行したりするために呼び出せる、`@tool`で装飾されたPython関数。

以下では、人気のある旅行先のリストを返すシンプルなツールを定義します。ユーザーが旅行のおすすめを求めたときに、エージェントはこのツールを使用します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ストリーミングレスポンス

よりインタラクティブな体験のために、エージェントの応答を<strong>ストリーミング</strong>できます。完全な応答を待つのではなく、エージェントは生成されるテキストのチャンクを逐次出力します。これは、リアルタイムで出力を表示したいチャットインターフェースで特に有用です。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 概要

このレッスンでは以下を学びました：

- `FoundryChatClient` 経由で Microsoft Foundry Agent Service に接続する <strong>プロバイダーの作成</strong>。
- エージェントがあなたの Python 関数を呼び出せるように `@tool` デコレーターを使って <strong>ツールを定義</strong>。
- ユーザーメッセージで <strong>エージェントを実行</strong> してその応答を表示。
- リアルタイム出力のための <strong>レスポンスのストリーム処理</strong>。

次のレッスンでは、エージェントフレームワークをより深く探り、エージェントにより強力なツールや複数ステップ推論機能を持たせる方法を学びます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
